# GeoLife CP1 — Same-second × Transportation Audit

**Mục tiêu:** kiểm chứng assumption Stage 2 sau góp ý mentor: `max_radius_m > 10 m` có luôn là spatial corruption/conflict, hay một phần là **timestamp-resolution ambiguity** trong các mode tốc độ cao.

Notebook này **không thay đổi production cleaning**. Nó chỉ tạo evidence để quyết định có amend contract hay không.

## Audit này đang kiểm tra điều gì?

GeoLife chỉ lưu timestamp tới whole second. Hai observations thật có thể xảy ra ở hai thời điểm khác nhau bên trong cùng một giây nhưng vẫn được ghi cùng timestamp:

```text
thời gian thật              timestamp trong release
12:00:00.10  P1   ┐
                  ├──→      12:00:00  P1
12:00:00.90  P2   ┘         12:00:00  P2
```

Ta không biết P1 hay P2 xảy ra trước từ release này.

Audit không hỏi “hai điểm >10m có chắc chắn hợp lệ không?”. Nó hỏi câu hẹp hơn:

> **Có evidence cho thấy một số group >10m có thể tương thích với chuyển động thật, nên không được mặc định gọi tất cả chúng là corruption hay không?**

Flow:

```text
212,409 same-second groups
        ↓
835 groups có max_radius_m > 10m
        ↓
reconstruct exact diameter
        ↓
join transportation labels theo [start, end)
        ↓
so sánh spatial spread theo mode
        ↓
so với distance-scale quan sát được từ speed benchmark
        ↓
quyết định semantics của threshold 10m
```

Transportation labels chỉ dùng như **audit evidence**, không trở thành runtime dependency của cleaning.

> **Quan trọng:** airplane là motivating example của mentor, nhưng audit chỉ có thể kết luận trực tiếp về airplane nếu trong 835 groups thực sự có airplane label unambiguous.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from time import perf_counter
from IPython.display import display
import os
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_URL = "https://github.com/tanh1c/geolife.git"
REPO_BRANCH = os.environ.get("GEOLIFE_REPO_BRANCH", "cp1-cleaning-staypoint")
REPO_DIR = Path(os.environ.get("GEOLIFE_REPO_DIR", "/tmp/geolife"))
VOLUME_ROOT = Path("/mnt/geolife-data")
AUDIT_CACHE_DIR = VOLUME_ROOT / "cache" / "cp1_same_second_transport"
AUDIT_CACHE_DIR.mkdir(parents=True, exist_ok=True)

def resolve_data_root():
    env_root = os.environ.get("GEOLIFE_DATA_ROOT")
    candidates = ([Path(env_root)] if env_root else []) + [
        VOLUME_ROOT / "extracted" / "Geolife Trajectories 1.3" / "Data",
        VOLUME_ROOT / "Data",
    ]
    for candidate in candidates:
        if candidate.is_dir() and any(candidate.glob("*/Trajectory/*.plt")):
            return candidate
    for candidate in sorted(VOLUME_ROOT.glob("**/Data")):
        if candidate.is_dir() and any(candidate.glob("*/Trajectory/*.plt")):
            return candidate
    raise FileNotFoundError("GeoLife Data folder not found")

def ensure_repo():
    if (REPO_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

DATA_ROOT = resolve_data_root()
ensure_repo()
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from notebooks.eda_core import (
    inventory_dataset,
    read_plt,
    read_labels,
    haversine_vectorized,
)

print("Repo:", REPO_DIR)
print("Branch:", REPO_BRANCH)
print("Data root:", DATA_ROOT)
print("Audit cache:", AUDIT_CACHE_DIR)

## 1. Reuse same-second EDA cache

EDA trước đã đo **212,409 same-second groups**, trong đó **835 groups có max_radius_m > 10 m**.

Ta reuse cache đó thay vì đọc lại toàn release.

In [ ]:
preferred = VOLUME_ROOT / "eda_cache" / "same_second_groups_v1.pkl"
if preferred.exists():
    SAME_SECOND_CACHE = preferred
else:
    matches = sorted(VOLUME_ROOT.glob("**/same_second_groups_v1.pkl"))
    if not matches:
        raise FileNotFoundError(
            "Không tìm thấy same_second_groups_v1.pkl. "
            "Cần volume/cache từ notebook EDA trước."
        )
    SAME_SECOND_CACHE = matches[0]

same_second = pd.read_pickle(SAME_SECOND_CACHE).copy()
same_second["timestamp"] = pd.to_datetime(same_second["timestamp"], utc=True)

conflicts = same_second.loc[same_second["max_radius_m"] > 10.0].copy()
conflicts = conflicts.sort_values(
    ["user_id", "trajectory_id", "timestamp"], kind="stable"
).reset_index(drop=True)

print("Loaded:", SAME_SECOND_CACHE)
print("Same-second groups:", f"{len(same_second):,}")
print("Groups >10m:", f"{len(conflicts):,}")
display(conflicts[["group_size","unique_positions","max_radius_m","median_radius_m"]]
        .describe(percentiles=[.5,.75,.9,.95,.99]))

assert len(same_second) == 212_409, "EDA cache count differs from audited reference"
assert len(conflicts) == 835, "Conflict count differs from audited reference"

### Cách đọc tail >10m

Trong 835 groups của run đã audit:

- median `max_radius_m` ≈ **15.8m** → phần lớn chỉ vừa vượt 10m;
- p90 `diameter_m` ≈ **388.7m**;
- p95 `diameter_m` nhảy lên khoảng **617 km**;
- max `diameter_m` ≈ **852 km**.

Tail này rất heterogeneous: vài chục mét và hàng trăm km không nên bị diễn giải như cùng một phenomenon.

Ta tách hai câu hỏi:

```text
Q1. Có safe để collapse không?   → threshold 10m
Q2. Nguyên nhân spread là gì?    → audit; thường không xác định chắc chắn
```

## 2. Reconstruct exact group diameter for the 835 conflicts

EDA trước dùng `max_radius_m`: khoảng cách xa nhất từ raw point tới median center.

Ví dụ:

```text
P1 -------- M -------- P2
      20m       20m
```

thì `max_radius_m ≈ 20m`.

Để hỏi “hai observations xa nhau bao nhiêu trong cùng recorded second?”, ta thêm:

```text
diameter_m = max pairwise Haversine distance inside the group
           ≈ 40m trong ví dụ trên
```

Notebook chỉ đọc lại các trajectory chứa 835 groups >10m; không scan lại toàn bộ release.

In [ ]:
inventory, all_files = inventory_dataset(DATA_ROOT)
file_map = {(user_id, path.stem): path for user_id, path in all_files}

DIAMETER_CACHE = AUDIT_CACHE_DIR / "same_second_conflict_diameter_v1.pkl"

def exact_group_diameter(group):
    coords = group[["latitude", "longitude"]].drop_duplicates().to_numpy(dtype=float)
    if len(coords) < 2:
        return 0.0
    best = 0.0
    for i in range(len(coords) - 1):
        d = haversine_vectorized(
            np.full(len(coords) - i - 1, coords[i,0]),
            np.full(len(coords) - i - 1, coords[i,1]),
            coords[i+1:,0],
            coords[i+1:,1],
        )
        if len(d):
            best = max(best, float(np.nanmax(d)))
    return best

if DIAMETER_CACHE.exists():
    conflict_detail = pd.read_pickle(DIAMETER_CACHE)
    print("Loaded:", DIAMETER_CACHE)
else:
    rows = []
    grouped_targets = conflicts.groupby(["user_id", "trajectory_id"], sort=False)
    t0 = perf_counter()

    for idx, ((user_id, trajectory_id), targets) in enumerate(grouped_targets, 1):
        path = file_map[(str(user_id), str(trajectory_id))]
        raw = read_plt(path)
        target_times = set(pd.to_datetime(targets["timestamp"], utc=True))

        subset = raw.loc[
            raw["timestamp"].isin(target_times),
            ["timestamp", "latitude", "longitude"],
        ]

        by_ts = {ts: g for ts, g in subset.groupby("timestamp", sort=False)}

        for row in targets.itertuples(index=False):
            group = by_ts.get(row.timestamp)
            if group is None or group.empty:
                raise RuntimeError(f"Cannot reconstruct {user_id}/{trajectory_id} {row.timestamp}")
            rows.append({
                "user_id": str(user_id),
                "trajectory_id": str(trajectory_id),
                "timestamp": row.timestamp,
                "group_size": int(row.group_size),
                "unique_positions": int(row.unique_positions),
                "max_radius_m": float(row.max_radius_m),
                "median_radius_m": float(row.median_radius_m),
                "diameter_m": exact_group_diameter(group),
            })

        if idx % 100 == 0:
            print(f"{idx:,}/{len(grouped_targets):,} affected trajectories | "
                  f"{(perf_counter()-t0)/60:.1f} min")

    conflict_detail = pd.DataFrame(rows)
    conflict_detail.to_pickle(DIAMETER_CACHE)
    print("Saved:", DIAMETER_CACHE)

print("Conflict rows:", len(conflict_detail))
display(conflict_detail[["max_radius_m","diameter_m"]]
        .describe(percentiles=[.5,.75,.9,.95,.99]))
assert len(conflict_detail) == 835

## 3. Transportation labels với half-open [start, end)

Ta dùng cùng convention đã audit trước:

```
start <= timestamp < end
```

Nếu tại một timestamp có nhiều **distinct active modes**, group được đánh `ambiguous` và không ép thành một mode.

Nếu user có labels nhưng timestamp nằm ngoài mọi label window: `labeled_user_unlabeled_time`.

Nếu user hoàn toàn không có `labels.txt`: `unlabeled_user`.

In [ ]:
label_parts = []
for user_dir in sorted(p for p in DATA_ROOT.iterdir() if p.is_dir() and p.name.isdigit()):
    label_path = user_dir / "labels.txt"
    if label_path.exists():
        part = read_labels(label_path, user_dir.name)
        if not part.empty:
            label_parts.append(part)

labels_all = (
    pd.concat(label_parts, ignore_index=True)
    if label_parts
    else pd.DataFrame(columns=["start_time","end_time","mode","user_id"])
)
label_users = set(labels_all["user_id"].astype(str))

print("Label intervals:", f"{len(labels_all):,}")
print("Users with labels:", len(label_users))
display(labels_all["mode"].value_counts())

In [ ]:
def active_modes_at_timestamp(user_labels, timestamp):
    active = user_labels.loc[
        (user_labels["start_time"] <= timestamp)
        & (timestamp < user_labels["end_time"]),
        "mode",
    ]
    return tuple(sorted(set(active.astype(str))))

labels_by_user = {
    str(user_id): g.sort_values(["start_time","end_time"], kind="stable").reset_index(drop=True)
    for user_id, g in labels_all.groupby("user_id")
}

assigned_rows = []
for row in conflict_detail.itertuples(index=False):
    user_id = str(row.user_id)
    if user_id not in label_users:
        status = "unlabeled_user"
        mode = None
        active_modes = ()
    else:
        modes = active_modes_at_timestamp(labels_by_user[user_id], row.timestamp)
        active_modes = modes
        if len(modes) == 1:
            status = "unambiguous_mode"
            mode = modes[0]
        elif len(modes) > 1:
            status = "ambiguous_label"
            mode = None
        else:
            status = "labeled_user_unlabeled_time"
            mode = None

    assigned_rows.append({
        **row._asdict(),
        "label_status": status,
        "mode": mode,
        "active_modes": active_modes,
    })

audit = pd.DataFrame(assigned_rows)
AUDIT_CACHE = AUDIT_CACHE_DIR / "same_second_transport_audit_v1.pkl"
audit.to_pickle(AUDIT_CACHE)
print("Saved:", AUDIT_CACHE)

display(audit["label_status"].value_counts(dropna=False).rename("groups"))
display(
    audit.loc[audit["label_status"] == "unambiguous_mode", "mode"]
    .value_counts()
    .rename("groups")
)

### Cách đọc output transportation-label join

Run hiện tại account đủ **835/835 groups**:

| label_status | groups | Ý nghĩa |
|---|---:|---|
| `unambiguous_mode` | **403** | timestamp nằm trong đúng 1 distinct active mode; dùng được cho audit theo mode |
| `labeled_user_unlabeled_time` | **349** | user có labels nhưng timestamp ngoài mọi label interval |
| `unlabeled_user` | **83** | user không có transportation labels |

Khoảng **48.3% (403/835)** conflicts có mode đủ rõ để phân tích.

403 unambiguous groups gồm:

```text
walk      194
bike      145
subway     25
bus        19
taxi       10
car         7
train       3
airplane    0
```

Không có airplane case unambiguous, nên airplane vẫn là motivating theoretical example chứ chưa phải observed evidence từ 835 groups. Train/subway là fast-transport evidence quan sát được gần nhất.

## 4. Distribution của spatial spread theo mode

Đây là phần trả lời trực tiếp câu hỏi mentor.

Ta xem cả `max_radius_m` và `diameter_m`. Các mode có sample rất nhỏ phải được đọc như case evidence, không phải population estimate.

In [ ]:
labeled = audit[audit["label_status"] == "unambiguous_mode"].copy()

mode_summary = (
    labeled.groupby("mode")
    .agg(
        groups=("mode","size"),
        median_radius_m=("max_radius_m","median"),
        p90_radius_m=("max_radius_m", lambda s: s.quantile(.90)),
        p99_radius_m=("max_radius_m", lambda s: s.quantile(.99)),
        max_radius_m=("max_radius_m","max"),
        median_diameter_m=("diameter_m","median"),
        p90_diameter_m=("diameter_m", lambda s: s.quantile(.90)),
        max_diameter_m=("diameter_m","max"),
    )
    .sort_values("groups", ascending=False)
)
display(mode_summary)

if not labeled.empty:
    plot_df = labeled.copy()
    plot_df["log10_diameter_m"] = np.log10(plot_df["diameter_m"].clip(lower=0.1))
    ax = plot_df.boxplot(column="log10_diameter_m", by="mode", rot=45, figsize=(10,5))
    ax.set_title("Same-second conflict diameter by unambiguous transportation mode")
    ax.set_ylabel("log10(diameter_m)")
    plt.suptitle("")
    plt.show()

### Cách đọc `mode_summary` và boxplot

Boxplot dùng `log10(diameter_m)` vì diameter có heavy tail.

**Train:** 3 groups, median diameter ≈ **30.08m**, max ≈ **31.66m**. Đây là case evidence rằng một >10m group không bắt buộc phải là corruption; legitimate movement + whole-second timestamp quantization là một explanation khả dĩ. Sample chỉ n=3 nên không được coi là population estimate.

**Subway:** 25 groups, median diameter ≈ **19.07m**, p90 ≈ **38.59m**, max ≈ **89.80m**. Phần lớn khá nhỏ nhưng vài case lớn hơn ordinary subway-motion scale, nên population này là mixture.

**Walk/bike:** median diameter lần lượt ≈ **19.9m / 22.5m**, nhưng có cases tới **314m / 266m**. Ordinary movement không thể giải thích toàn bộ tail.

Vì vậy không được suy:

```text
>10m → fast movement
```

và cũng không được suy:

```text
>10m → corruption
```

Evidence hỗ trợ câu hẹp hơn:

```text
>10m → nguyên nhân chưa xác định, không còn safe để collapse
```

## 5. Spatial bins × mode

Bins giúp phân biệt “vừa vượt 10m” với conflict hàng km:

- 10–25m
- 25–50m
- 50–100m
- 100–333.3m
- 333.3m–1km
- >1km

`333.3m` = khoảng cách trong 1 giây ở hard guard 1200 km/h. Đây **không phải threshold mới**; chỉ là diagnostic envelope.

In [ ]:
bins = [10, 25, 50, 100, 333.333333, 1000, np.inf]
labels = ["10-25m","25-50m","50-100m","100-333m","333m-1km",">1km"]

audit["diameter_bin"] = pd.cut(
    audit["diameter_m"],
    bins=bins,
    labels=labels,
    right=True,
    include_lowest=False,
)

mode_or_status = audit["mode"].where(
    audit["label_status"] == "unambiguous_mode",
    audit["label_status"],
)
audit["mode_or_status"] = mode_or_status

display(pd.crosstab(audit["mode_or_status"], audit["diameter_bin"], margins=True))

under_guard = audit["diameter_m"] <= (1200.0 / 3.6)
print("All >10m conflict groups:", len(audit))
print("Diameter <=333.3m:", int(under_guard.sum()), f"({under_guard.mean():.2%})")

fast_modes = {"airplane","train","subway"}
fast = audit[
    (audit["label_status"] == "unambiguous_mode")
    & audit["mode"].isin(fast_modes)
].copy()

print("Unambiguous airplane/train/subway groups:", len(fast))
if len(fast):
    display(
        fast.sort_values("diameter_m", ascending=False)[
            ["user_id","trajectory_id","timestamp","mode",
             "group_size","max_radius_m","diameter_m"]
        ].head(50)
    )

### Cách đọc spatial bins và mốc 333.3m

`333.3m` đến từ hard-speed guard:

```text
1200 km/h / 3.6 = 333.3 m/s
```

Run hiện tại có **747/835 = 89.46%** groups với diameter <=333.3m. Đây **không** có nghĩa 89.46% groups là valid; nó chỉ là broad engineering envelope.

Ví dụ walk diameter 300m vẫn nằm dưới 333.3m nhưng không thể giải thích bằng ordinary walking trong <1s.

Ngược lại, **88/835** groups >333.3m, gồm 20 groups ở 333m–1km và 68 groups >1km; đây là strong spatial-inconsistency candidates.

Hai threshold trả lời hai câu hỏi khác nhau:

```text
10m     → đủ compact để safe-to-collapse không?
333.3m  → broad diagnostic envelope dưới hard guard
```

333.3m tuyệt đối không phải consolidation threshold mới.

## 6. Diagnostic compatibility với audited transport-speed evidence

Ta đổi speed reference từ V3 benchmark sang distance trong 1 giây:

| mode | reference | approx distance / 1s |
|---|---:|---:|
| airplane | observed max 1048.11 km/h | 291.1m |
| train | p99 210.34 km/h | 58.4m |
| taxi | p99 104.75 km/h | 29.1m |
| subway | p99 94.32 km/h | 26.2m |
| car | p99 119.63 km/h | 33.2m |
| bus | p99 90.59 km/h | 25.2m |
| bike | p99 40.63 km/h | 11.3m |
| walk | p99 40.39 km/h | 11.2m |

Reference này chỉ là diagnostic context, không phải validity rule.

In [ ]:
reference_kmh = {
    "airplane": 1048.11,  # observed max in audited V3
    "train": 210.34,      # p99
    "taxi": 104.75,       # p99
    "subway": 94.32,      # p99
    "car": 119.63,        # p99
    "bus": 90.59,         # p99
    "bike": 40.63,        # p99
    "walk": 40.39,        # p99; diagnostic only
}

diag = labeled[labeled["mode"].isin(reference_kmh)].copy()
diag["reference_kmh"] = diag["mode"].map(reference_kmh)
diag["reference_1s_m"] = diag["reference_kmh"] / 3.6
diag["within_reference_1s"] = diag["diameter_m"] <= diag["reference_1s_m"]

compat = (
    diag.groupby("mode")
    .agg(
        groups=("mode","size"),
        reference_1s_m=("reference_1s_m","first"),
        within_reference_groups=("within_reference_1s","sum"),
        within_reference_rate=("within_reference_1s","mean"),
        median_diameter_m=("diameter_m","median"),
        max_diameter_m=("diameter_m","max"),
    )
)
display(compat)

### Cách diễn giải compatibility table

Run đầy đủ:

- **train:** 3/3 = **100%** nằm trong 1-second reference scale;
- **subway:** 20/25 = **80%**;
- **taxi:** 7/10 = **70%**;
- **bus:** 10/19 ≈ **52.6%**;
- **car:** 3/7 ≈ **42.9%**;
- **bike:** 2/145 ≈ **1.4%**;
- **walk:** 4/194 ≈ **2.1%**.

Pattern này là evidence rằng population >10m là **heterogeneous mixture**:

```text
train / nhiều subway cases
→ movement + timestamp quantization là explanation khả dĩ

walk / bike phần lớn cases
→ ordinary movement không đủ để giải thích spread
```

Transportation mode giúp audit mixture đó; production cleaning không được dùng mode để repair/order points.

## 7. Kết luận audit và implication cho cleaning contract

### Kết luận chính

> **10m là safe-to-collapse threshold, không phải valid-vs-corrupt threshold.**

```text
same-second max_radius <= 10m
→ đủ compact
→ safe to consolidate bằng robust median representative

same-second max_radius > 10m
→ không còn safe để collapse
→ within-second ordering không identifiable
→ continuity boundary
→ không tự động kết luận nguyên nhân là corruption
```

### Evidence dẫn tới kết luận

1. 403/835 groups có transportation mode unambiguous.
2. Không có airplane case unambiguous; airplane vẫn là motivating example.
3. Train: 3/3 groups nằm trong 1-second reference scale.
4. Subway: 20/25 groups nằm trong ~26.2m 1-second p99 scale, nhưng một số case lớn tới ~90m.
5. Walk/bike có nhiều >10m cases ordinary movement không giải thích được.
6. 68 groups có diameter >1km và là strong spatial-inconsistency candidates.

Do đó `>10m` là mixture of possible causes; wording “corruption” cho toàn bộ tail là quá mạnh.

### Điều audit không chứng minh

Audit không chứng minh rằng mọi group >10m là valid, là fast movement, hoặc threshold nên tăng lên 50/100/333m. Transportation labels cũng không trở thành runtime feature.

### Contract implication

Diagnostic reason nên dùng tên trung tính:

```text
same_second_spatial_ambiguity
```

Production behavior giữ nguyên:

```text
<=10m → median-collapse
>10m  → discard unresolved timestamp from retained sequence + continuity boundary
```

Lý do boundary là **ordering không xác định**, không phải vì ta chắc chắn data corrupt.

Stay-point baseline không cần rerun vì sequence-boundary behavior không đổi.